In [ ]:
import os

# Fix wandb — tránh bị hỏi interactive login
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_START_METHOD"] = "thread"

# Clone repo và cài đặt
!git clone https://github.com/minhvuongle2004/lung-diagnosis.git
%cd /kaggle/working/lung-diagnosis/ldct-benchmark
!pip install -e . -q

print("✅ Setup done!")

In [ ]:
import os

# Tự động tìm thư mục cha chứa LDCT-and-Projection-data
print("=== /kaggle/input/ contents ===")
for item in os.listdir("/kaggle/input"):
    print(f"  {item}/")

data_path = None
print("\n=== Tìm LDCT-and-Projection-data ===")
for dataset_slug in os.listdir("/kaggle/input"):
    base = f"/kaggle/input/{dataset_slug}"
    for root, dirs, files in os.walk(base):
        if "LDCT-and-Projection-data" in dirs:
            data_path = root
            sub = os.path.join(root, "LDCT-and-Projection-data")
            patients = sorted(os.listdir(sub))
            print(f"✅ Datafolder: {data_path}")
            print(f"   Số bệnh nhân: {len(patients)}")
            print(f"   5 đầu: {patients[:5]}")
            break
    if data_path:
        break

if data_path is None:
    print("❌ Không tìm thấy LDCT-and-Projection-data!")

In [ ]:
import os, yaml

# Đọc info.yml để biết dataset yêu cầu những bệnh nhân nào
info_path = "ldctbench/data/info.yml"
with open(info_path) as f:
    info = yaml.safe_load(f)

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
available = set(os.listdir(ldct_dir))

print(f"Bệnh nhân có trong dataset: {len(available)}")
print(f"Bệnh nhân có sẵn: {sorted(available)}\n")

missing_patients = []
missing_files = []

for split in ["train_set", "val_set", "test_set"]:
    for entry in info.get(split, []):
        pid = entry["id"]
        if pid not in available:
            missing_patients.append((split, pid))
        else:
            # Kiểm tra folder input tồn tại
            input_rel = entry["input"].replace("./LDCT-and-Projection-data/", "")
            input_full = os.path.join(ldct_dir, input_rel)
            if not os.path.exists(input_full):
                missing_files.append((split, pid, input_full))

if missing_patients:
    print(f"❌ Bệnh nhân THIẾU ({len(missing_patients)}):")
    for split, pid in missing_patients:
        print(f"   [{split}] {pid}")
else:
    print("✅ Tất cả bệnh nhân đều có mặt!")

if missing_files:
    print(f"\n❌ Folder THIẾU ({len(missing_files)}):")
    for split, pid, path in missing_files[:10]:
        print(f"   [{split}] {pid}: {path}")
else:
    print("✅ Tất cả folder DICOM đều tồn tại!")

In [ ]:
import os, yaml, shutil

# Tự động tạo info_filtered.yml chỉ gồm các bệnh nhân CÓ SẴN trong dataset
info_path = "ldctbench/data/info.yml"
with open(info_path) as f:
    info = yaml.safe_load(f)

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
available = set(os.listdir(ldct_dir))

new_info = {k: v for k, v in info.items() if k not in ["train_set", "val_set", "test_set"]}

for split in ["train_set", "val_set", "test_set"]:
    original = info.get(split, [])
    filtered = [e for e in original if e["id"] in available]
    new_info[split] = filtered
    removed = len(original) - len(filtered)
    print(f"{split}: {len(original)} → {len(filtered)} bệnh nhân (bỏ {removed})")

# Backup info.yml gốc và ghi đè bằng filtered
shutil.copy(info_path, info_path + ".bak")
with open(info_path, "w") as f:
    yaml.dump(new_info, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Đã lưu info.yml mới (backup: info.yml.bak)")
print("   Sẵn sàng train với bệnh nhân có sẵn!")

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"

assert data_path is not None, "❌ Chạy Cell 2 trước!"
print(f"✅ Datafolder: {data_path}")

config = f"""trainer: edrrednet
seed: 1339
datafolder: {data_path}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.1
loss_beta: 0.0
loss_gamma: 0.0
num_edge_blocks: 2
mbs: 16
max_iterations: 92994
data_subset: 1.0
patchsize: 128
iterations_before_val: 500
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

with open("configs/edrrednet_kaggle.yaml", "w", encoding="utf-8") as f:
    f.write(config)

print("Config saved. Bắt đầu training seed 1339...")
!python -m ldctbench.scripts.train --config configs/edrrednet_kaggle.yaml

In [ ]:
import glob, shutil, os

output_dir = "/kaggle/working"
seed = 1339

checkpoints = glob.glob("wandb/latest-run/*.pt")
print(f"Checkpoints found: {checkpoints}")
for ckpt in checkpoints:
    dest = os.path.join(output_dir, f"seed{seed}_{os.path.basename(ckpt)}")
    shutil.copy(ckpt, dest)
    print(f"✅ Saved: {dest}")

logs = glob.glob("wandb/latest-run/files/*.csv")
for log in logs:
    dest = os.path.join(output_dir, f"seed{seed}_{os.path.basename(log)}")
    shutil.copy(log, dest)
    print(f"✅ Log: {dest}")

print("\n✅ Done! Click 'Save Version' để lưu output.")